# E04 · YOLO 道路目标检测与视觉感知探索 V1.0 DEV
先预测、再操作、最后解释。5 个核心实验，素材 A/B/C 已通过教师审核。使用 YOLO26n/s COCO 预训练模型，不训练。结果用于教学，不用于驾驶决策。当前 Notebook 为未完成实机验收的研发初稿。

准备：在隔离环境中安装 `ultralytics==8.4.219`、OpenCV、Matplotlib、NumPy、PyTorch（保留已有 CUDA 配置）。将审核 ZIP 解包至工作目录，使 `dist/frames`、`dist/videos`、`dist/data` 可用；Kaggle 可上传同一素材包及模型作为数据集。模型权重从官方渠道获取；联网受限时选择已有权重路径。

In [ ]:
from pathlib import Path
import hashlib, json, time
import cv2, numpy as np, matplotlib.pyplot as plt, torch
from ultralytics import YOLO
import ultralytics
ROOT = Path("dist")  # 如在 Kaggle，修改为实际素材目录
WEIGHTS = Path("models")
assert ultralytics.__version__ == "8.4.219", "请使用已验证素材包的版本，避免输出模式差异"
assert (ROOT/"frames/A1.jpg").is_file(), "请确认素材路径"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
HASHES = {"yolo26n":"9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef","yolo26s":"646f8bc3fe0a656803d95c294f7852321748cb29d13466a1af8862e2db384a1b"}
models={}
def get_model(name):
    if name not in models:
        p=WEIGHTS/(name+".pt")
        assert p.is_file(), f"Missing official weights {p}"
        assert hashlib.sha256(p.read_bytes()).hexdigest()==HASHES[name], "权重与审核基线不同"
        models[name]=YOLO(str(p))
    return models[name]
def image(key):
    im=cv2.imread(str(ROOT/"frames"/(key+".jpg")))
    assert im is not None, key
    return im
def predict(im, model="yolo26n", size=640, conf=.25, iou=.5, traditional=True):
    t=time.perf_counter()
    m=get_model(model)
    out=m.predict(im,imgsz=size,conf=conf,iou=iou,nms=traditional,rect=False,device=DEVICE,verbose=False)[0]
    assert bool(m.predictor.model.end2end)==(not traditional), "检测头模式与预期不一致"
    return out,(time.perf_counter()-t)*1000
def show(im, configs):
    fig,axes=plt.subplots(1,len(configs)+1,figsize=(5*(len(configs)+1),4),squeeze=False)
    axes[0,0].imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB)); axes[0,0].set_title("Original")
    for ax,cfg in zip(axes[0,1:],configs):
        r,ms=predict(im,**cfg)
        ax.imshow(cv2.cvtColor(r.plot(),cv2.COLOR_BGR2RGB));ax.set_title(f"{cfg}\n{len(r.boxes)} boxes, {ms:.0f} ms")
    for ax in axes.flat: ax.axis("off")
    plt.tight_layout();plt.show()
print("Device:",DEVICE)

## 01 · 基础目标检测
先数画面中的车辆和行人，再运行模型。图像分类告诉我们有什么；目标检测还告诉我们在哪里。切换 A1、B3，比较类别与位置。检测分数不是校准概率。

In [ ]:
show(image("A1"),[{}])
show(image("B3"),[{}])

## 02 · 置信度
置信度阈值像录取分数线。提高会过滤低分预测，不保证更准确。先预测 C1 的车在哪个阈值消失，再观察四组结果；检查是否误删真目标。

In [ ]:
show(image("C1"),[dict(conf=x) for x in [.05,.25,.5,.8]])
show(image("B3"),[dict(conf=x) for x in [.05,.25,.5,.8]])

## 03 · IoU 与 NMS
两个框的交集面积除以并集面积。NMS 对同类候选优先保留高分框；阈值越低，抑制越强。先用几何示例理解，再在 B1、B3 的真实模型结果中改变 IoU。免 NMS 分支不应因 IoU 改变。

In [ ]:
def box_iou(a,b):
    a,b=np.asarray(a),np.asarray(b)
    inter=np.maximum(0,np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])).prod()
    union=np.maximum(0,a[2:]-a[:2]).prod()+np.maximum(0,b[2:]-b[:2]).prod()-inter
    return float(inter/union) if union else 0.
def greedy_nms(boxes, scores, classes, iou=.5):
    kept=[]
    for i in sorted(range(len(scores)),key=lambda i:(-scores[i],i)):
        if not any(classes[i]==classes[j] and box_iou(boxes[i],boxes[j])>iou for j in kept): kept.append(i)
    return kept
boxes=[[0,0,100,100],[10,10,110,110],[75,0,175,100]]
for t in [.05,.3,.5,.9]:
    print(t,greedy_nms(boxes,[.92,.81,.86],[2,2,2],t))
show(image("B1"),[dict(conf=.05,iou=t) for t in [.3,.5,.7]])
show(image("B1"),[dict(conf=.05,iou=t,traditional=False) for t in [.3,.7]])

## 04 · 输入分辨率与模型规模
输入分辨率好比望远镜，模型规模好比辨认物体的经验。控制变量：先固定模型比较输入尺寸，再固定尺寸比较模型。计时含调用开销，第一次加载与预热影响比较；检测框数不是准确率。

In [ ]:
show(image("A3"),[dict(size=z) for z in [320,640,960]])
show(image("A3"),[dict(model=m) for m in ["yolo26n","yolo26s"]])
show(image("C1"),[dict(model=m) for m in ["yolo26n","yolo26s"]])

## 05 · 连续视频中的目标检测
先按 2 Hz 抽样进行真实逐帧推理；不将其称为全帧实时检测。分析数量突然变化的时间点，回看原图判断目标是否真正消失。需要全帧时把 sample_fps 改成源 FPS，运行时间会增加。

In [ ]:
def process_clip(scene="B", sample_fps=2, model="yolo26n", size=640, conf=.25, iou=.5):
    cap=cv2.VideoCapture(str(ROOT/"videos"/(scene+".mp4")))
    assert cap.isOpened(), "无法读取视频"
    fps=cap.get(cv2.CAP_PROP_FPS); assert fps>0
    stride=max(1,round(fps/sample_fps))
    rows=[];i=0;first=None
    try:
        while True:
            ok,im=cap.read()
            if not ok: break
            if i%stride==0:
                r,ms=predict(im,model=model,size=size,conf=conf,iou=iou)
                counts={name:0 for name in ("car","person","bicycle","truck","bus")}
                for c in r.boxes.cls.tolist():
                    name=r.names[int(c)]
                    if name in counts: counts[name]+=1
                rows.append(dict(t=i/fps,ms=ms,**counts))
                if first is None: first=r.plot()
            i+=1
    finally: cap.release()
    assert rows, "无有效帧"
    for c in ("car","person","bicycle"):
        plt.plot([r["t"] for r in rows],[r[c] for r in rows],label=c)
    plt.xlabel("Source video time (s)");plt.ylabel("Detection count");plt.legend();plt.show()
    plt.imshow(cv2.cvtColor(first,cv2.COLOR_BGR2RGB));plt.axis("off");plt.show()
    return rows
rows=process_clip("B",sample_fps=2)

## 课堂总结
在 A/B/C 选择一个失败片段，记录时间、模型、尺寸、置信度与 IoU。每次只改变一个参数，解释观察结果。没有人工框真值，不能计算准确率或 mAP。素材作者、许可和改动说明详见 dist/data/sources.json 与 dist/licenses；A 保留 MIT，B/C 保留 CC BY 3.0 作者署名和链接。